In [ ]:
!pip install transformers
!pip install datasets
!pip install scikit-learn
!pip install pandas
!pip install torch

In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake.head()

In [ ]:
fake["label"] = 0
true["label"] = 1

In [ ]:
df = pd.concat([fake, true])
df.head()

In [ ]:
df.shape

In [ ]:
df["label"].value_counts()

In [ ]:
df["content"] = df["title"] + " " + df["text"]

In [ ]:
df[["content","label"]].head()

In [ ]:
!pip install transformers
!pip install torch
!pip install scikit-learn

In [ ]:
import torch
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
tokens = tokenizer(
    df["content"].tolist(),
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

In [ ]:
tokens.keys()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    tokens["input_ids"],
    df["label"],
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
from transformers import BertForSequenceClassification
import torch

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(device)

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
os.listdir()

In [ ]:
import pandas as pd

fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

In [ ]:
fake["label"] = 0
true["label"] = 1

In [ ]:
df = pd.concat([fake, true])

In [ ]:
df["content"] = df["title"] + " " + df["text"]

In [ ]:
import pandas as pd
from transformers import BertTokenizer

# Load the datasets
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

# Add labels to the datasets
fake["label"] = 0
true["label"] = 1

# Concatenate the datasets
df = pd.concat([fake, true])

# Create the 'content' column
df["content"] = df["title"] + " " + df["text"]

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

tokens = tokenizer(
    df["content"].tolist(),
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    tokens["input_ids"],
    df["label"],
    test_size=0.2,
    random_state=42
)

In [ ]:
import torch

y_train_tensor = torch.tensor(y_train.values)
y_test_tensor = torch.tensor(y_test.values)

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

X = np.array(tokens["input_ids"])
y = np.array(df["label"])

x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
import torch

X_train = torch.tensor(x_train)
X_test = torch.tensor(x_test)

y_train_tensor = torch.tensor(y_train)
y_test_tensor = torch.tensor(y_test)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train_tensor)
val_dataset = TensorDataset(X_test, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(device)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
epochs = 2

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in train_loader:

        input_ids, labels = batch

        input_ids = input_ids.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print("Epoch:", epoch+1, "Loss:", total_loss/len(train_loader))

In [ ]:
from sklearn.metrics import classification_report

model.eval()

predictions = []
true_labels = []

for batch in val_loader:

    input_ids, labels = batch

    input_ids = input_ids.to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids)

    logits = outputs.logits
    preds = torch.argmax(logits, dim=1)

    predictions.extend(preds.cpu().numpy())
    true_labels.extend(labels.numpy())

print(classification_report(true_labels, predictions))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(true_labels, predictions)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
for i in range(len(predictions)):
    if predictions[i] != true_labels[i]:
        print("Predicted:", predictions[i])
        print("Actual:", true_labels[i])
        print()

In [ ]:
count = 0

for i in range(len(predictions)):
    if predictions[i] != true_labels[i]:
        print("Predicted:", predictions[i])
        print("Actual:", true_labels[i])
        print()

        count += 1
        if count == 5:
            break